# 06 - Model Comparison

Trains all 9 supervised classifiers, compares them on the held-out test set
using Accuracy / Precision / Recall / F1 / ROC-AUC / PR-AUC / MCC / Log-Loss.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src.utils import get_config, setup_logging
from src.models import ClassificationTrainer
from src.evaluation import evaluate_all, select_best
setup_logging()
cfg = get_config()

# Load processed splits
X_tr = pd.read_csv(cfg.processed_data_dir / "X_train.csv")
y_tr = pd.read_csv(cfg.processed_data_dir / "y_train.csv").iloc[:,0]
X_te = pd.read_csv(cfg.processed_data_dir / "X_test.csv")
y_te = pd.read_csv(cfg.processed_data_dir / "y_test.csv").iloc[:,0]

# Apply FE + FS
from src.features import FeatureEngineer, FeatureSelector
eng = FeatureEngineer.load(cfg.preprocessing_dir / "feature_engineer.joblib")
sel = __import__("joblib").load(cfg.features_dir / "feature_selector.joblib")
X_tr_e = sel.transform(eng.transform(X_tr))
X_te_e = sel.transform(eng.transform(X_te))
print(f"Train: {X_tr_e.shape}  Test: {X_te_e.shape}")


## 1. Train all models

In [ ]:
trainer = ClassificationTrainer(params_cfg=cfg.params, model_cfg=cfg.model)
results = trainer.train_all(X_tr_e, y_tr, cfg.trained_models_dir)
print(f"Trained {len(results)} models")


## 2. Evaluate on test set

In [ ]:
eval_results, cmp_df = evaluate_all(results, X_te_e, y_te, cfg.evaluations_dir)
cmp_df.sort_values('recall_phishing', ascending=False)


## 3. Best model

In [ ]:
best = select_best(eval_results)
print(f"Best: {best.name}")
print(f"  recall_phishing={best.recall_phishing:.4f}  roc_auc={best.roc_auc:.4f}")
print(f"  accuracy={best.accuracy:.4f}  f1={best.f1:.4f}  mcc={best.mcc:.4f}")
print(f"  confusion_matrix (TP,FN,FP,TN)={best.confusion_matrix}")


## 4. Plot metric comparison

In [ ]:
import matplotlib.pyplot as plt
metrics = ['accuracy','precision','recall','f1','roc_auc','pr_auc','mcc']
fig, ax = plt.subplots(figsize=(12, 6))
cmp_df.set_index('model')[metrics].plot(kind='bar', ax=ax, colormap='viridis')
ax.set_title('Model comparison - all metrics')
ax.set_ylabel('Score'); ax.legend(loc='lower right', fontsize=8)
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()


## Summary

All 9 base classifiers + 2 ensembles train successfully on the engineered
feature set. The best model is chosen by **recall on the phishing class**
(since missing a phishing site is costlier than a false alarm), with
ROC-AUC as tiebreaker. The next notebook trains AutoGluon and compares
against the manual ensemble.
